# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [1]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

Fichero: C:\Users\alvar\Master_Comillas\Operacion_Modelos\practicas\icai-muiaap-operacion-de-modelos\semana3\assets\03-wine-quality\inference_samples.csv
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Filas de ejemplo: 5


## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Evita tener que reentrenar el modelo en cada inferencia; encapsula los pesos aprendidos y la lógica matemática para predecir (`predict` / `predict_proba`). |
| Versión del esquema | **manifest.json** (`schema_version`) | Evita cargar un bundle cuya estructura o especificación de metadatos no sea comprendida por el consumidor o cargador actual. |
| Orden y nombres de features | **manifest.json** (`feature_names`) | Evita la corrupción silenciosa de datos (*silent data corruption*); garantiza que las columnas del vector de entrada coincidan en nombre, cantidad y orden exacto con las que aprendió el estimador. |
| Versión del preprocesado | **manifest.json** (`preprocessing_version`) | Evita discrepancias entre el vectorizado/transformaciones en tiempo de servicio y las usadas durante el entrenamiento. |
| Etiquetas de salida permitidas | **manifest.json** (`output_labels`) | Evita que el modelo retorne clases no contempladas por el contrato del consumidor aguas abajo (dashboard, API REST). |
| Versión del modelo | **manifest.json** (`model_version`) | Evita la pérdida de trazabilidad, linaje y auditoría; identifica unívocamente el experimento y artefacto desplegado. |
| Tipo de estimador | **manifest.json** (`estimator_type`) | Permite inspeccionar qué algoritmo y clase (`RandomForestClassifier`, etc.) se empaquetó sin necesidad de deserializar el binario, comprobando compatibilidad de interfaz. |

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

### Decisiones ante anomalías:

1. **¿Qué debe pasar si falta `manifest.json`?**  
   Se detiene inmediatamente el proceso elevando `FileNotFoundError` o `ValueError` antes de deserializar el estimador. Sin manifiesto no hay contrato ni garantía de compatibilidad.
2. **¿Qué debe pasar si aparecen features desconocidas o en un orden distinto?**  
   El cargador debe rechazar el artefacto elevando `ValueError` al cotejar `manifest.feature_names` contra `FEATURE_NAMES`, bloqueando la inferencia para impedir predicciones erróneas e indetectables (*silent corruption*).
3. **¿Qué debe pasar si el modelo devuelve una etiqueta que no está en `output_labels`?**  
   La función de inferencia (`infer_wine_quality`) debe interceptarla y lanzar un `ValueError`, garantizando que solo salidas válidas según contrato lleguen al usuario o API.
4. **¿Qué debe pasar si una fila del CSV no cumple el contrato de entrada?**  
   Pydantic (`WineQualityRequest`) eleva un `ValidationError` (propagado como `ValueError`). El CLI detiene la ejecución inmediatamente y no produce ninguna salida parcial (escritura atómica).


In [2]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

manifest_draft = {
    "schema_version": "wine-quality-bundle-v1",
    "model_version": legacy_model_version,
    "preprocessing_version": "wine-red-features-v1",
    "feature_names": feature_names,
    "output_labels": ["needs_review", "acceptable", "excellent"],
    "estimator_type": "RandomForestClassifier",
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.")

{
  "schema_version": "wine-quality-bundle-v1",
  "model_version": "wine-quality-rf-demo-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "needs_review",
    "acceptable",
    "excellent"
  ],
  "estimator_type": "RandomForestClassifier"
}

La estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.


## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | Validación de manifiesto (`load_model_bundle` / `ArtifactManifest`) | `ValueError: feature_names no coincide con el orden del contrato.` | No |
| Cambia `preprocessing_version` | Validación de manifiesto (`load_model_bundle` / `ArtifactManifest`) | `ValueError: preprocessing_version incompatible: se esperaba wine-red-features-v1.` | No |
| El modelo devuelve `unknown` | Validación de salida (`infer_wine_quality`) | `ValueError: Etiqueta de inferencia desconocida o no permitida: unknown.` | No |
| La petición incluye `ph = 99` | Validación de contrato de entrada (`WineQualityRequest`) | `ValueError / ValidationError: Input should be less than or equal to 4.5.` | No |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- **¿Qué campos son obligatorios y cuáles tienen valores por defecto?**  
  Todos los seis campos del manifiesto (`schema_version`, `model_version`, `preprocessing_version`, `feature_names`, `output_labels`, `estimator_type`) son estrictamente obligatorios. No se permiten valores por defecto que puedan ocultar configuraciones implícitas.
- **¿Aceptarás campos desconocidos en el manifiesto?**  
  No. Se configura `ConfigDict(extra="forbid")` en Pydantic para rechazar taxativamente campos no declarados y prevenir drift no detectado.
- **¿Validarás el manifiesto antes de cargar el objeto serializado?**  
  Sí, siempre se valida `manifest.json` antes de cargar `model.joblib`. Esto previene deserializaciones innecesarias de binarios incompatibles y mitiga riesgos de seguridad.
- **¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?**  
  Se procesan todas las filas acumulándolas en memoria; el CSV solo se escribe si el 100% de las filas se validaron e infirieron sin errores (salida atómica todo-o-nada).
- **¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?**  
  Un mensaje descriptivo y accionable que especifique la fila, identificador de muestra, columna o versión causante del error, finalizando con un código de salida no nulo.
- **¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?**  
  Se interrumpe la ejecución elevando el error y descartando las predicciones previas. No se crea ningún CSV ni se publica una salida parcial.

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.